# Cost-Optimal Data Center Self-Supply Under Curtailment

This notebook builds and solves the continuous capacity-planning and event-dispatch model in `draft_model.tex`.

**Implementation assumptions:** scenarios are independent events with the resource energy state reset to full at each event; each period is one hour; capacity costs are linear annualized costs; `sigma` is fixed exogenous load shifting; and unserved load is nonnegative. Network constraints and inter-event energy carryover are outside the draft model.

**Prerequisites:** the active Julia environment must have `JuMP` and `HiGHS` installed (`import Pkg; Pkg.add(["JuMP", "HiGHS"])` if needed).


In [2]:
using JuMP
using HiGHS


## Input data contract

The model consumes a Julia `NamedTuple` named `data`. Values must be normalized to MW, MWh, hours, and consistent currency units. A MATPOWER `.m` case can provide candidate power limits; scenario profiles and additional technology attributes must be joined/preprocessed into this interface. No project data or paths are assumed.

| Field | Shape | Meaning |
|---|---:|---|
| `p_max`, `e_max` | generators | Maximum buildable MW and MWh |
| `p_cap_cost`, `e_cap_cost` | generators | Annualized cost per MW and MWh |
| `marginal_cost`, `emissions_rate`, `eta` | generators | Dispatch cost, emissions per MWh, and discharge efficiency |
| `availability` | scenarios × generators | `alpha[omega,g]` in `[0,1]` |
| `scenario_weight`, `duration` | scenarios | Scenario weight and number of one-hour periods |
| `demand`, `kappa`, `sigma` | scenarios × padded periods | Baseline MW, curtailment fraction, and fixed MW shift; periods beyond `duration[omega]` are ignored |
| `carbon_price`, `unserved_penalty` | scalars | Carbon price and penalty per unserved MWh |

`theta` and `u` are nonnegative. `sigma` is treated as given data because the draft does not define it as a decision variable. Cost curves are linearized as `p_cap_cost[g] * P[g]` and `e_cap_cost[g] * E[g]`.


In [3]:
function validate_data(data)
    required = (
        :p_max, :e_max, :p_cap_cost, :e_cap_cost, :marginal_cost,
        :emissions_rate, :eta, :availability, :scenario_weight, :duration,
        :demand, :kappa, :sigma, :carbon_price, :unserved_penalty,
    )
    for field in required
        hasproperty(data, field) || throw(ArgumentError("missing data field: $field"))
    end

    ng = length(data.p_max)
    nω = length(data.duration)
    ng > 0 || throw(ArgumentError("at least one generator candidate is required"))
    nω > 0 || throw(ArgumentError("at least one scenario is required"))
    nt = size(data.demand, 2)

    for field in (:e_max, :p_cap_cost, :e_cap_cost, :marginal_cost, :emissions_rate, :eta)
        length(getproperty(data, field)) == ng ||
            throw(ArgumentError("$field must have one value per generator"))
    end
    length(data.scenario_weight) == nω ||
        throw(ArgumentError("scenario_weight must have one value per scenario"))
    size(data.availability) == (nω, ng) ||
        throw(ArgumentError("availability must have shape (scenarios, generators)"))
    for field in (:kappa, :sigma)
        size(getproperty(data, field)) == size(data.demand) ||
            throw(ArgumentError("$field must have the same shape as demand"))
    end
    nt > 0 || throw(ArgumentError("at least one padded time period is required"))
    all(t -> 1 <= t <= nt, data.duration) ||
        throw(ArgumentError("each scenario duration must be between 1 and the padded horizon"))

    for field in (:p_max, :e_max, :p_cap_cost, :e_cap_cost, :marginal_cost,
                  :emissions_rate, :scenario_weight)
        all(x -> isfinite(x) && x >= 0, getproperty(data, field)) ||
            throw(ArgumentError("$field values must be finite and nonnegative"))
    end
    all(x -> isfinite(x) && 0 < x <= 1, data.eta) ||
        throw(ArgumentError("eta values must be in (0, 1]"))
    all(x -> isfinite(x) && 0 <= x <= 1, data.availability) ||
        throw(ArgumentError("availability values must be in [0, 1]"))
    all(x -> isfinite(x) && x >= 0, data.demand) ||
        throw(ArgumentError("demand values must be finite and nonnegative"))
    all(x -> isfinite(x) && 0 <= x <= 1, data.kappa) ||
        throw(ArgumentError("kappa values must be in [0, 1]"))
    all(x -> isfinite(x) && x >= 0, data.sigma) ||
        throw(ArgumentError("sigma values must be finite and nonnegative"))
    isfinite(data.carbon_price) && data.carbon_price >= 0 ||
        throw(ArgumentError("carbon_price must be finite and nonnegative"))
    isfinite(data.unserved_penalty) && data.unserved_penalty >= 0 ||
        throw(ArgumentError("unserved_penalty must be finite and nonnegative"))

    for ω in 1:nω, t in 1:data.duration[ω]
        data.sigma[ω, t] <= data.kappa[ω, t] * data.demand[ω, t] ||
            throw(ArgumentError("sigma cannot exceed required demand reduction at scenario $ω, period $t"))
    end
    return nothing
end

function build_model(data; optimizer = HiGHS.Optimizer)
    validate_data(data)
    ng = length(data.p_max)
    nω = length(data.duration)
    nt = size(data.demand, 2)

    model = Model(optimizer)
    set_silent(model)

    @variable(model, P[1:ng] >= 0)
    @variable(model, E[1:ng] >= 0)
    @variable(model, theta[1:ng, 1:nω, 1:nt] >= 0)
    @variable(model, u[1:nω, 1:nt] >= 0)
    @variable(model, bE[1:ng, 1:nω, 1:(nt + 1)] >= 0)

    @constraint(model, [g in 1:ng], P[g] <= data.p_max[g])
    @constraint(model, [g in 1:ng], E[g] <= data.e_max[g])

    for ω in 1:nω
        for g in 1:ng
            @constraint(model, bE[g, ω, 1] == E[g])
            for t in 1:(data.duration[ω] + 1)
                @constraint(model, bE[g, ω, t] <= E[g])
            end
        end
        for t in 1:data.duration[ω]
            @constraint(model,
                sum(theta[g, ω, t] for g in 1:ng) + data.sigma[ω, t] + u[ω, t] ==
                data.kappa[ω, t] * data.demand[ω, t])
            for g in 1:ng
                @constraint(model, theta[g, ω, t] <= data.availability[ω, g] * P[g])
                @constraint(model,
                    bE[g, ω, t + 1] - bE[g, ω, t] == -theta[g, ω, t] / data.eta[g])
            end
        end
    end

    @objective(model, Min,
        sum(data.p_cap_cost[g] * P[g] + data.e_cap_cost[g] * E[g] for g in 1:ng) +
        sum(data.scenario_weight[ω] * (
                sum((data.marginal_cost[g] + data.carbon_price * data.emissions_rate[g]) *
                    theta[g, ω, t] for g in 1:ng) +
                data.unserved_penalty * u[ω, t])
            for ω in 1:nω for t in 1:data.duration[ω]))

    return model, (P = P, E = E, theta = theta, u = u, bE = bE)
end

function solve_model(data; optimizer = HiGHS.Optimizer)
    model, variables = build_model(data; optimizer)
    optimize!(model)
    has_values(model) || error("No feasible solution returned (status: $(termination_status(model)))")

    return (
        termination_status = termination_status(model),
        objective = objective_value(model),
        power_capacity_mw = value.(variables.P),
        energy_capacity_mwh = value.(variables.E),
        dispatch_mw = value.(variables.theta),
        unserved_mw = value.(variables.u),
        state_of_charge_mwh = value.(variables.bE),
    )
end


solve_model (generic function with 1 method)

## Synthetic smoke test

The following two-candidate, one-event data are illustrative only. Replace them with project inputs after preprocessing to the schema above. The checks verify event power balance and non-increasing state of charge under discharge.


In [4]:
data = (
    p_max = [3.0, 4.0],
    e_max = [6.0, 8.0],
    p_cap_cost = [40.0, 80.0],
    e_cap_cost = [10.0, 15.0],
    marginal_cost = [5.0, 20.0],
    emissions_rate = [0.0, 0.5],
    eta = [0.9, 0.9],
    availability = [1.0 1.0],
    scenario_weight = [1.0],
    duration = [2],
    demand = [8.0 8.0],
    kappa = [0.5 0.5],
    sigma = [0.0 0.0],
    carbon_price = 50.0,
    unserved_penalty = 1000.0,
)


(p_max = [3.0, 4.0], e_max = [6.0, 8.0], p_cap_cost = [40.0, 80.0], e_cap_cost = [10.0, 15.0], marginal_cost = [5.0, 20.0], emissions_rate = [0.0, 0.5], eta = [0.9, 0.9], availability = [1.0 1.0], scenario_weight = [1.0], duration = [2], demand = [8.0 8.0], kappa = [0.5 0.5], sigma = [0.0 0.0], carbon_price = 50.0, unserved_penalty = 1000.0)

In [5]:
results = solve_model(data)

println("Termination status: ", results.termination_status)
println("Objective: ", results.objective)
println("Built power capacity (MW): ", results.power_capacity_mw)
println("Built energy capacity (MWh): ", results.energy_capacity_mwh)
println("Dispatch by generator, scenario, period (MW):
", results.dispatch_mw)
println("Unserved load by scenario and period (MW):
", results.unserved_mw)

for t in 1:data.duration[1]
    required_reduction = data.kappa[1, t] * data.demand[1, t]
    delivered = sum(results.dispatch_mw[g, 1, t] for g in eachindex(data.p_max)) +
                data.sigma[1, t] + results.unserved_mw[1, t]
    @assert isapprox(delivered, required_reduction; atol = 1e-7)
end
for g in eachindex(data.p_max), t in 1:(data.duration[1] + 1)
    @assert -1e-7 <= results.state_of_charge_mwh[g, 1, t] <= results.energy_capacity_mwh[g] + 1e-7
end
println("Smoke-test balance and energy-state checks passed.")


Termination status: OPTIMAL
Objective: 459.33333333333337
Built power capacity (MW): [2.6999999999999997, 1.3000000000000003]
Built energy capacity (MWh): [6.0, 2.8888888888888897]
Dispatch by generator, scenario, period (MW):
[2.6999999999999997; 1.3000000000000003;;; 2.6999999999999997; 1.3000000000000003]
Unserved load by scenario and period (MW):
[0.0 0.0]
Smoke-test balance and energy-state checks passed.


## Scope and next steps

- `P[g]` and `E[g]` are continuous build decisions; `theta[g, omega, t]` is available self-supply, and `u[omega, t]` is unserved curtailment.
- The objective uses linear capacity costs and the draft's weighted event operating cost, including carbon cost `(marginal_cost + carbon_price * emissions_rate)`.
- Period durations are one hour, so the state equation uses `theta / eta` without an additional duration factor. For other interval lengths, multiply the state change by interval hours.
- Energy-state variables are independently initialized to the built energy capacity for each scenario, matching the draft's full initial condition and standalone-event assumption.
- `sigma` has no cost or decision constraints in the draft; it is therefore an exogenous relief profile here. Define it as a decision variable only after specifying its economics and limits.
